In [1]:
from Simulation.mpc import *
from Simulation.system_functions import PolymerCSTR
from utils.helpers import *

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
dir_path = os.path.join(os.getcwd(), "Data")

In [8]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[3.2, 321],
                       [4.5, 325]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [9]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [10]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [11]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [12]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [13]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [14]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

In [15]:
# Setpoints in deviation form
inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324],
                          [3.4, 321]])

y_sp_scenario = (apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
                 - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]))
n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False, False, False, False]
warm_start = 5
ACTOR_FREEZE = 0 * set_points_len
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

In [16]:
# Observer Gain
poles = np.array(np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262 , 0.96916776, 0.91230187]))
L = compute_observer_gain(A_aug, C_aug, poles)

The system is observable.


C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Simulation\mpc.py:123: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999999422182038.
  obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method='KNV0')


## Setting The hyperparameters for the TD3 Agent


In [17]:
from TD3Agent.agent import TD3Agent
from DQN.dqn_agent import DQNAgent
import torch

# Model Multipliers Agent

In [18]:
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM = 3 # or Matrix A nad B (Change this to a more general format)
n_outputs = C_aug.shape[0]
a_coef, b_coef, c_coef = 1, 1, 1
ACTOR_LAYER_SIZES = [512, 512, 512]
CRITIC_LAYER_SIZES = [512, 512, 512]
HIGH_COEF = np.array([1.05, 1.05, 1.05])
LOW_COEF = np.array([0.95, 0.95, 0.95])
model_low = LOW_COEF          # for (alpha, delta1, delta2)
model_high = HIGH_COEF
BUFFER_CAPACITY = 25000
ACTOR_LR = 1e-4
CRITIC_LR = 1e-4
SMOOTHING_STD = 0.1
NOISE_CLIP = 0.2
GAMMA = 0.995
TAU = 0.005
MAX_ACTION = 1.0
POLICY_DELAY = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
STD_START = 0.2
STD_END = 0.02
STD_DECAY_RATE = 0.99995
STD_DECAY_MODE = "exp"
ACTOR_FREEZE = 0

In [19]:
td3_agent_model = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    actor_freeze=ACTOR_FREEZE,
    )

# Weights update Agent

In [20]:
set_points_number = int(C_aug.shape[0])
inputs_number = int(B_aug.shape[1])
STATE_DIM = int(A_aug.shape[0]) + set_points_number + inputs_number
ACTION_DIM=4
HIGH_COEF=np.array([3.0,3.0,3.0,3.0])
LOW_COEF=np.array([0.5,0.5,0.5,0.5])
weights_low = LOW_COEF        # for (Q1, Q2, R1, R2) multipliers in your weights run
weights_high = HIGH_COEF
n_outputs = C_aug.shape[0]
ACTOR_LAYER_SIZES = [512, 512, 512, 512, 512]
CRITIC_LAYER_SIZES = [512, 512, 512, 512, 512]
BUFFER_CAPACITY = 50000
ACTOR_LR = 1e-4
CRITIC_LR = 1e-4
SMOOTHING_STD = 0.1
NOISE_CLIP = 0.2
GAMMA = 0.995
TAU = 0.005
MAX_ACTION = 1.0
POLICY_DELAY = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
STD_START = 0.2
STD_END = 0.02
STD_DECAY_RATE = 0.9999
STD_DECAY_MODE = "exp"
ACTOR_FREEZE = 0

In [21]:
td3_agent_weights = TD3Agent(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    actor_hidden=ACTOR_LAYER_SIZES,
    critic_hidden=CRITIC_LAYER_SIZES,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    batch_size=BATCH_SIZE,
    policy_delay=POLICY_DELAY,
    target_policy_smoothing_noise_std=SMOOTHING_STD,
    noise_clip=NOISE_CLIP,
    max_action=MAX_ACTION,
    tau=TAU,
    std_start=STD_START,
    std_end=STD_END,
    std_decay_rate=STD_DECAY_RATE,
    std_decay_mode=STD_DECAY_MODE,
    buffer_size=BUFFER_CAPACITY,
    device=DEVICE,
    actor_freeze=ACTOR_FREEZE,
    )

# Horizon Agent

In [22]:
PREDICT_GRID = list(range(8, 20))  # Hp candidates
CONTROL_GRID = list(range(3, 10))     # Hc candidates (will be filtered by Hc <= Hp)

In [23]:
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
N_ACTIONS = len(HORIZON_RECIPES)

In [24]:
STATE_DIM = int(A_aug.shape[0]) + int(C_aug.shape[0]) + int(B_aug.shape[1])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
HIDDEN_LAYERS = [512, 512, 512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4
INPUT_NUMBER = B_aug.shape[1]

Using device: cuda


In [25]:
dqn_agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=N_ACTIONS,
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update="soft",
    tau=0.01,
    hard_update_interval=10_000,
    target_combine="q1",
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.3,
    eps_end=0.01,
    eps_decay_rate=0.99999,
    eps_decay_mode="exp",
)

## MPC initialization and setpoints

In [26]:
# MPC parameters
n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states['ss_inputs'], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78]), data_min[:n_inputs], data_max[:n_inputs])
b_max= apply_min_max(np.array([870, 670]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0]-u_ss[0], b_max[0]-u_ss[0])
b2 = (b_min[1]-u_ss[1], b_max[1]-u_ss[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(n_inputs*cont_h)
Q1_penalty = 5.
Q2_penalty = 1.
R1_penalty = 1
R2_penalty = 1
Q_base = np.array([Q1_penalty, Q2_penalty], dtype=float)
R_base = np.array([R1_penalty, R2_penalty], dtype=float)

In [27]:
MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty]),
    R_in=np.array([R1_penalty, R2_penalty]),
    NP=predict_h,
    NC=cont_h)

In [28]:
def make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=5.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
):
    """
    Reward with relative tracking bands.

    data_min, data_max : arrays for [u_min..., y_min...], [u_max..., y_max...]
    n_inputs           : number of inputs (so outputs start at index n_inputs)
    k_rel              : per-output relative tolerance factors (same length as outputs)
    band_floor_phys    : per-output minimum band in physical units
    Q_diag, R_diag     : quadratic weights (same as before)
    """

    data_min = np.asarray(data_min, float)
    data_max = np.asarray(data_max, float)
    dy = np.maximum(data_max[n_inputs:] - data_min[n_inputs:], 1e-12)  # phys range for each y

    k_rel = np.asarray(k_rel, float)
    band_floor_phys = np.asarray(band_floor_phys, float)
    Q_diag = np.asarray(Q_diag, float)
    R_diag = np.asarray(R_diag, float)

    # floor in *scaled* coordinates (used if y_sp_phys is not provided)
    band_floor_scaled = band_floor_phys / np.maximum(dy, 1e-12)

    def _sigmoid(x):
        x = np.clip(x, -60.0, 60.0)
        return 1.0 / (1.0 + np.exp(-x))

    def _phi(z, kind=bonus_kind, k=bonus_k, p=bonus_p, c=bonus_c):
        z = np.clip(z, 0.0, 1.0)
        if kind == "linear":
            return 1.0 - z
        if kind == "quadratic":
            return (1.0 - z) ** 2
        if kind == "exp":
            return (np.exp(-k * z) - np.exp(-k)) / (1.0 - np.exp(-k))
        if kind == "power":
            return 1.0 - np.power(z, p)
        if kind == "log":
            return np.log1p(c * (1.0 - z)) / np.log1p(c)
        raise ValueError("unknown bonus kind")

    def reward_fn(e_scaled, du_scaled, y_sp_phys=None):
        """
        e_scaled : output error in scaled deviation space  (same as before)
        du_scaled: input move in scaled deviation space    (same as before)
        y_sp_phys: current setpoint in *physical* units (array len = n_outputs)
        """

        e_scaled = np.asarray(e_scaled, float)
        du_scaled = np.asarray(du_scaled, float)

        # ----- dynamic band based on setpoint -----
        if y_sp_phys is None:
            # fallback: just use the floor
            band_scaled = band_floor_scaled
        else:
            y_sp_phys_arr = np.asarray(y_sp_phys, float)
            # band_phys_i = max(k_rel_i * |y_sp_i|, band_floor_phys_i)
            band_phys = np.maximum(k_rel * np.abs(y_sp_phys_arr), band_floor_phys)
            band_scaled = band_phys / np.maximum(dy, 1e-12)

        tau_scaled = tau_frac * band_scaled

        # ----- inside/outside gate -----
        abs_e = np.abs(e_scaled)
        s_i = _sigmoid((band_scaled - abs_e) / np.maximum(tau_scaled, 1e-12))

        if gate == "prod":
            w_in = float(np.prod(s_i, dtype=np.float64))
        elif gate == "mean":
            w_in = float(np.mean(s_i))
        elif gate == "geom":
            w_in = float(np.prod(s_i, dtype=np.float64) ** (1.0 / len(s_i)))
        else:
            raise ValueError("gate must be 'prod'|'mean'|'geom'")

        # ----- core quadratic costs -----
        err_quad = np.sum(Q_diag * (e_scaled ** 2))
        err_eff = (1.0 - w_in) * err_quad + w_in * (lam_in * err_quad)
        move = np.sum(R_diag * (du_scaled ** 2))

        # ----- linear penalties around band edge -----
        slope_at_edge = 2.0 * Q_diag * band_scaled

        overflow = np.maximum(abs_e - band_scaled, 0.0)
        lin_out = (1.0 - w_in) * np.sum(gamma_out * slope_at_edge * overflow)

        inside_mag = np.minimum(abs_e, band_scaled)
        lin_in = w_in * np.sum(gamma_in * slope_at_edge * inside_mag)

        # ----- bonus near zero error -----
        qb2 = Q_diag * (band_scaled ** 2)
        z = abs_e / np.maximum(band_scaled, 1e-12)
        phi = _phi(z)
        bonus = w_in * beta * np.sum(qb2 * phi)

        # ----- total reward -----
        return (-(err_eff + move + lin_out + lin_in) + bonus) * 0.01

    params = dict(
        k_rel=k_rel,
        band_floor_phys=band_floor_phys,
        band_floor_scaled=band_floor_scaled,
        Q_diag=Q_diag,
        R_diag=R_diag,
        tau_frac=tau_frac,
        gamma_out=gamma_out,
        gamma_in=gamma_in,
        beta=beta,
        gate=gate,
        lam_in=lam_in,
        bonus_kind=bonus_kind,
        bonus_k=bonus_k,
        bonus_p=bonus_p,
        bonus_c=bonus_c,
    )
    return params, reward_fn

In [29]:
n_inputs = 2

dy = data_max[n_inputs:] - data_min[n_inputs:]
y_sp_nom = 0.5 * (data_min[n_inputs:] + data_max[n_inputs:])

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])

band_phys = np.maximum(k_rel * np.abs(y_sp_nom), band_floor_phys)

scale_factor = 1.0  # use 2.0 for [-1, 1] scaling, 1.0 for [0, 1]
band_scaled = scale_factor * band_phys / dy

q0 = 1.4
Q_diag = q0 / np.maximum(band_scaled ** 2, 1e-12)

print("dy:", dy)
print("y_sp_nom:", y_sp_nom)
print("band_phys:", band_phys)
print("band_scaled:", band_scaled)
print("Q_diag:", Q_diag)

dy: [0.22165278 0.78153727]
y_sp_nom: [  3.83915067 323.21371982]
band_phys: [0.01151745 0.09696412]
band_scaled: [0.05196169 0.12406845]
Q_diag: [518.51529284  90.95055189]


In [30]:
Q_diag = np.array([518., 90.])          # rounded from the band-based calculation
R_diag = np.array([90., 90.])          # move cost for du_scaled ~ 0.02

n_inputs = 2

print("Band scaled are:")

params, reward_fn = make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=7.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
)
print(params)

Band scaled are:
{'k_rel': array([0.003 , 0.0003]), 'band_floor_phys': array([0.006, 0.07 ]), 'band_floor_scaled': array([0.02706937, 0.08956707]), 'Q_diag': array([518.,  90.]), 'R_diag': array([90., 90.]), 'tau_frac': 0.7, 'gamma_out': 0.5, 'gamma_in': 0.5, 'beta': 7.0, 'gate': 'geom', 'lam_in': 1.0, 'bonus_kind': 'exp', 'bonus_k': 12.0, 'bonus_p': 0.6, 'bonus_c': 20.0}


In [31]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

In [32]:
import warnings
def compute_observer_gain(A, C, desired_poles):
    """
    Compute an observer gain L for the given MPC system using the desired poles.
    Also performs an observability check.

    Parameters:
    -----------
    A, C : np.ndarray
        System Matrices
    desired_poles : np.ndarray
        A vector of desired observer poles.

    Returns:
    --------
    L : np.ndarray
        The observer gain matrix.
    """
    # Compute the observer gain using pole placement
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, "KNV0")
    L = np.squeeze(obs_gain_calc.gain_matrix).T

    # Check observability
    observability_matrix = control.obsv(A, C)
    rank = np.linalg.matrix_rank(observability_matrix)
    if rank == A.shape[0]:
        # print("The system is observable.")
        pass
    else:
        # print("The system is not observable.")
        pass
    return L

In [33]:
import numpy as np
import scipy.optimize as spo

def run_multi_agent_rl_mpc(
    system,
    y_sp_scenario,
    n_tests,
    set_points_len,
    steady_states,
    min_max_dict,
    agent_model,      # TD3 for alpha, delta (can be None)
    agent_horizon,    # DQN for (Hp, Hc) (can be None)
    agent_weights,    # TD3 for [Q1, Q2, R1, R2] multipliers (can be None)
    A_aug,
    B_aug,
    C_aug,
    Q_base,
    R_base,
    poles,
    data_min,
    data_max,
    warm_start,
    test_cycle,
    horizon_recipes=None,
    mpc_default_horizons=(9, 3),
    decision_interval=4,
    model_low=None,
    model_high=None,
    weights_low=None,
    weights_high=None,
    reward_fn=None,
    mode="disturb",
    nominal_qi=None,
    nominal_qs=None,
    nominal_ha=None,
    qi_change=None,
    qs_change=None,
    ha_change=None,
    observer_update_interval=25,
    done_on_setpoint_change=False,
    verbose=True,
):
    """
    Combined supervisor:
      - agent_model: TD3 outputs raw action in [-1,1], mapped to [model_low, model_high]
        expected dims: 1 + n_inputs  (alpha + delta vector)
      - agent_horizon: DQN outputs discrete index selecting (Hp, Hc) from horizon_recipes
      - agent_weights: TD3 outputs raw action in [-1,1], mapped to [weights_low, weights_high]
        expected dims: 4 (Q1, Q2, R1, R2 multipliers)
    """

    if reward_fn is None:
        raise ValueError("reward_fn must be provided")

    if agent_horizon is not None and horizon_recipes is None:
        raise ValueError("horizon_recipes must be provided when agent_horizon is not None")

    # --- setpoints and (optional) disturbances ---
    y_sp, nFE, sub_episodes_changes_dict, time_in_sub_episodes, test_train_dict, WARM_START, qi, qs, ha = \
        generate_setpoints_training_rl_gradually(
            y_sp_scenario, n_tests, set_points_len, warm_start, test_cycle,
            nominal_qi, nominal_qs, nominal_ha,
            qi_change, qs_change, ha_change
        )

    n_inputs = B_aug.shape[1]
    n_outputs = C_aug.shape[0]
    n_states = A_aug.shape[0]

    ss_scaled_inputs = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
    y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[n_inputs:], data_max[n_inputs:])

    # absolute scaled actuator limits
    b_min_abs = apply_min_max(np.array([71.6, 78.0]), data_min[:n_inputs], data_max[:n_inputs])
    b_max_abs = apply_min_max(np.array([870.0, 670.0]), data_min[:n_inputs], data_max[:n_inputs])

    # decision variables are du, so bounds must be in deviation form
    b_min_dev = b_min_abs - ss_scaled_inputs
    b_max_dev = b_max_abs - ss_scaled_inputs
    b1 = (float(b_min_dev[0]), float(b_max_dev[0]))
    b2 = (float(b_min_dev[1]), float(b_max_dev[1]))
    cons = []

    # immutable baselines (avoid cumulative drift)
    A_base = A_aug.copy()
    B_base = B_aug.copy()

    Q_base = np.asarray(Q_base, float).reshape(-1)
    R_base = np.asarray(R_base, float).reshape(-1)
    if Q_base.size != n_outputs:
        raise ValueError("Q_base must have length n_outputs")
    if R_base.size != n_inputs:
        raise ValueError("R_base must have length n_inputs")

    # helpers
    def map_to_bounds(a, low, high):
        a = np.asarray(a, float)
        return low + ((a + 1.0) / 2.0) * (high - low)

    def map_from_bounds(x, low, high):
        x = np.asarray(x, float)
        return 2.0 * (x - low) / (high - low) - 1.0

    # defaults when agents are absent
    if model_low is None:
        model_low = np.ones(1 + n_inputs) * 1.0
    if model_high is None:
        model_high = np.ones(1 + n_inputs) * 1.0
    model_low = np.asarray(model_low, float).reshape(-1)
    model_high = np.asarray(model_high, float).reshape(-1)

    if weights_low is None:
        weights_low = np.ones(4) * 1.0
    if weights_high is None:
        weights_high = np.ones(4) * 1.0
    weights_low = np.asarray(weights_low, float).reshape(-1)
    weights_high = np.asarray(weights_high, float).reshape(-1)

    # baseline raw actions (so mapped values equal 1.0 exactly)
    model_baseline_raw = np.zeros(1 + n_inputs, dtype=float)
    weights_baseline_raw = map_from_bounds(np.ones(4, dtype=float), weights_low, weights_high)

    # horizon baseline index
    default_Hp, default_Hc = mpc_default_horizons
    if horizon_recipes is not None:
        if (default_Hp, default_Hc) in horizon_recipes:
            horizon_baseline_idx = int(horizon_recipes.index((default_Hp, default_Hc)))
        else:
            horizon_baseline_idx = 0
    else:
        horizon_baseline_idx = 0

    # logging
    y_system = np.zeros((nFE + 1, n_outputs))
    y_system[0, :] = system.current_output
    u_mpc = np.zeros((nFE, n_inputs))
    rewards = np.zeros(nFE)
    avg_rewards = []
    yhat = np.zeros((n_outputs, nFE))
    xhatdhat = np.zeros((n_states, nFE + 1))

    horizon_trace = np.zeros((nFE, 2), dtype=int)
    alpha_log = np.ones((nFE, 1), dtype=float)
    delta_log = np.ones((nFE, n_inputs), dtype=float)
    weight_log = np.ones((nFE, 4), dtype=float)

    # MPC object, rebuildable
    current_Hp, current_Hc = int(default_Hp), int(default_Hc)
    MPC_obj = MpcSolverGeneral(
        A_base.copy(), B_base.copy(), C_aug,
        Q_out=Q_base.copy(),
        R_in=R_base.copy(),
        NP=current_Hp,
        NC=current_Hc
    )

    def rebuild_mpc(Hp, Hc, A_now, B_now, Q_now, R_now):
        nonlocal MPC_obj
        MPC_obj = MpcSolverGeneral(
            A_now, B_now, C_aug,
            Q_out=Q_now,
            R_in=R_now,
            NP=int(Hp),
            NC=int(Hc)
        )

    # observer gain
    L = compute_observer_gain(MPC_obj.A, MPC_obj.C, poles)

    last_h_idx = None
    test = False

    for i in range(nFE):
        if i in test_train_dict:
            test = test_train_dict[i]

        scaled_current_input = apply_min_max(system.current_input, data_min[:n_inputs], data_max[:n_inputs])
        scaled_current_input_dev = scaled_current_input - ss_scaled_inputs

        s = apply_rl_scaled(min_max_dict, xhatdhat[:, i], y_sp[i, :], scaled_current_input_dev).astype(np.float32)

        # 1) horizon decision
        if agent_horizon is None:
            h_idx = horizon_baseline_idx
            Hp, Hc = current_Hp, current_Hc
        else:
            if i <= WARM_START:
                h_idx = horizon_baseline_idx
            else:
                if (i % decision_interval == 0) or (last_h_idx is None):
                    if not test:
                        h_idx = int(agent_horizon.take_action(s, eval_mode=False))
                    else:
                        h_idx = int(agent_horizon.act_eval(s))
                    last_h_idx = h_idx
                else:
                    h_idx = int(last_h_idx)

            Hp, Hc = action_to_horizons(horizon_recipes, h_idx)

        # 2) model scaling decision
        if agent_model is None:
            a_model_raw = model_baseline_raw
        else:
            if i <= WARM_START:
                a_model_raw = model_baseline_raw
            else:
                if not test:
                    a_model_raw = agent_model.take_action(s, explore=True)
                else:
                    a_model_raw = agent_model.act_eval(s)
        a_model_raw = np.asarray(a_model_raw, dtype=float).reshape(-1)

        m_model = map_to_bounds(a_model_raw, model_low, model_high)
        alpha = float(m_model[0])
        delta = np.asarray(m_model[1:1 + n_inputs], float).reshape(-1)

        # build A,B from baselines every step
        A_now = A_base.copy()
        B_now = B_base.copy()
        n_phys = n_states - n_outputs
        A_now[:n_phys, :n_phys] *= alpha
        B_now[:n_phys, :] *= delta.reshape(1, -1)

        # 3) weights decision
        if agent_weights is None:
            a_w_raw = weights_baseline_raw
        else:
            if i <= WARM_START:
                a_w_raw = weights_baseline_raw
            else:
                if not test:
                    a_w_raw = agent_weights.take_action(s, explore=True)
                else:
                    a_w_raw = agent_weights.act_eval(s)
        a_w_raw = np.asarray(a_w_raw, dtype=float).reshape(-1)
        if a_w_raw.size != 4:
            raise ValueError("agent_weights must output 4 actions for [Q1,Q2,R1,R2] multipliers")

        m_w = map_to_bounds(a_w_raw, weights_low, weights_high)
        Q_now = Q_base.copy()
        R_now = R_base.copy()
        Q_now[0] *= float(m_w[0])
        Q_now[1] *= float(m_w[1])
        R_now[0] *= float(m_w[2])
        R_now[1] *= float(m_w[3])

        # apply horizon change by rebuilding if needed
        if (int(Hp), int(Hc)) != (current_Hp, current_Hc):
            rebuild_mpc(Hp, Hc, A_now.copy(), B_now.copy(), Q_now.copy(), R_now.copy())
            current_Hp, current_Hc = int(Hp), int(Hc)

        # update MPC internals regardless
        MPC_obj.A = A_now
        MPC_obj.B = B_now
        MPC_obj.Q_out = Q_now
        MPC_obj.R_in = R_now

        horizon_trace[i, :] = (current_Hp, current_Hc)
        alpha_log[i, 0] = alpha
        delta_log[i, :] = delta
        weight_log[i, :] = m_w

        # # observer gain update sometimes (since A changed)
        # if (i % max(1, observer_update_interval) == 0) or (i == 0):
        #     L = compute_observer_gain(MPC_obj.A, MPC_obj.C, poles)

        # solve MPC
        bnds = (b1, b2) * current_Hc
        IC_opt = np.zeros(n_inputs * current_Hc)

        sol = spo.minimize(
            lambda x: MPC_obj.mpc_opt_fun(x, y_sp[i, :], scaled_current_input_dev, xhatdhat[:, i]),
            IC_opt,
            bounds=bnds,
            constraints=cons
        )

        u_mpc[i, :] = sol.x[:n_inputs] + ss_scaled_inputs
        u_plant = reverse_min_max(u_mpc[i, :], data_min[:n_inputs], data_max[:n_inputs])

        delta_u = u_mpc[i, :] - scaled_current_input

        system.current_input = u_plant
        system.step()
        if mode == "disturb":
            system.hA = ha[i]
            system.Qs = qs[i]
            system.Qi = qi[i]

        y_system[i + 1, :] = system.current_output

        # observer roll
        y_current_scaled = apply_min_max(y_system[i + 1, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled
        y_prev_scaled = apply_min_max(y_system[i, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled

        delta_y = y_current_scaled - y_sp[i, :]
        yhat[:, i] = MPC_obj.C @ xhatdhat[:, i]
        xhatdhat[:, i + 1] = (MPC_obj.A @ xhatdhat[:, i]
                              + MPC_obj.B @ (u_mpc[i, :] - ss_scaled_inputs)
                              + (L @ (y_prev_scaled - yhat[:, i])).T)

        y_sp_phys = reverse_min_max(y_sp[i, :] + y_ss_scaled, data_min[n_inputs:], data_max[n_inputs:])

        # IMPORTANT: do not multiply by 0.01 again if reward_fn already scales
        r = float(reward_fn(delta_y, delta_u, y_sp_phys))
        rewards[i] = r

        next_u_dev = u_mpc[i, :] - ss_scaled_inputs
        s2 = apply_rl_scaled(min_max_dict, xhatdhat[:, i + 1], y_sp[i, :], next_u_dev).astype(np.float32)

        if done_on_setpoint_change and (i in sub_episodes_changes_dict):
            done = 1.0
        else:
            done = 0.0

        # training
        if not test:
            if agent_model is not None and i > WARM_START:
                agent_model.push(s, a_model_raw.astype(np.float32), r, s2, float(done))
                if i >= warm_start:
                    agent_model.train_step()

            if agent_weights is not None and i > WARM_START:
                agent_weights.push(s, a_w_raw.astype(np.float32), r, s2, float(done))
                if i >= warm_start:
                    agent_weights.train_step()

            if agent_horizon is not None and i > WARM_START:
                agent_horizon.push(s, int(h_idx), r, s2, float(done))
                if i >= warm_start:
                    agent_horizon.train_step()

        # diagnostics
        if i in sub_episodes_changes_dict:
            avg_r = float(np.mean(rewards[max(0, i - time_in_sub_episodes + 1): i + 1]))
            avg_rewards.append(avg_r)

            if verbose:
                print("Sub_Episode:", sub_episodes_changes_dict[i], "| avg reward:", avg_r)
                print("Hp,Hc:", (current_Hp, current_Hc))
                print("alpha avg:", float(np.mean(alpha_log[max(0, i - time_in_sub_episodes + 1): i + 1, 0])))
                print("delta avg:", np.mean(delta_log[max(0, i - time_in_sub_episodes + 1): i + 1, :], axis=0))
                print("weights avg [Q1,Q2,R1,R2]:", np.mean(weight_log[max(0, i - time_in_sub_episodes + 1): i + 1, :], axis=0))

    u_rl = reverse_min_max(u_mpc, data_min[:n_inputs], data_max[:n_inputs])

    return (
        y_system,
        u_rl,
        avg_rewards,
        rewards,
        xhatdhat,
        nFE,
        time_in_sub_episodes,
        y_sp,
        yhat,
        horizon_trace,
        alpha_log,
        delta_log,
        weight_log
    )

In [34]:
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
out = run_multi_agent_rl_mpc(
    system=cstr,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    min_max_dict=min_max_dict,
    agent_model=td3_agent_model,       # your multipliers TD3
    agent_horizon=dqn_agent,           # your horizon DQN
    agent_weights=td3_agent_weights,   # your weights TD3
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Q_base=Q_base,
    R_base=R_base,
    poles=poles,
    data_min=data_min,
    data_max=data_max,
    warm_start=warm_start,
    test_cycle=TEST_CYCLE,
    horizon_recipes=HORIZON_RECIPES,
    mpc_default_horizons=(predict_h, cont_h),
    decision_interval=DECISION_INTERVAL,
    model_low=model_low,
    model_high=model_high,
    weights_low=LOW_COEF,
    weights_high=HIGH_COEF,
    reward_fn=reward_fn,
    mode="disturb",
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    observer_update_interval=25,
    done_on_setpoint_change=False,
    verbose=True
)
(y_system, u_rl, avg_rewards, rewards, xhatdhat, nFE, time_in_sub_episodes,
    y_sp, yhat, horizon_trace, alpha_log, delta_log, weight_log) = out

Sub_Episode: 1 | avg reward: -3.3676516384653956
Hp,Hc: (9, 3)
alpha avg: 1.0
delta avg: [1. 1.]
weights avg [Q1,Q2,R1,R2]: [1. 1. 1. 1.]
Sub_Episode: 2 | avg reward: -4.417427036157696
Hp,Hc: (9, 3)
alpha avg: 1.0
delta avg: [1. 1.]
weights avg [Q1,Q2,R1,R2]: [1. 1. 1. 1.]
Sub_Episode: 3 | avg reward: -4.412449880938803
Hp,Hc: (9, 3)
alpha avg: 1.0
delta avg: [1. 1.]
weights avg [Q1,Q2,R1,R2]: [1. 1. 1. 1.]
Sub_Episode: 4 | avg reward: -4.408853842440393
Hp,Hc: (9, 3)
alpha avg: 1.0
delta avg: [1. 1.]
weights avg [Q1,Q2,R1,R2]: [1. 1. 1. 1.]
Sub_Episode: 5 | avg reward: -4.405123900506683
Hp,Hc: (9, 3)
alpha avg: 1.0
delta avg: [1. 1.]
weights avg [Q1,Q2,R1,R2]: [1. 1. 1. 1.]
Sub_Episode: 6 | avg reward: -3.703417789073356
Hp,Hc: (18, 6)
alpha avg: 0.9934871837793209
delta avg: [0.98540798 0.97072369]
weights avg [Q1,Q2,R1,R2]: [1.84357084 2.25113382 2.33436279 1.71744859]
Sub_Episode: 7 | avg reward: -4.416675006970667
Hp,Hc: (9, 9)
alpha avg: 0.9942763249435128
delta avg: [0.9850593

In [35]:
from utils.plotting import plot_rl_results_multiagent_dqnstyle

In [36]:
out_dir = plot_rl_results_multiagent_dqnstyle(
    y_sp=y_sp,
    steady_states=steady_states,
    nFE=nFE,
    delta_t=delta_t,
    time_in_sub_episodes=time_in_sub_episodes,
    y_rl=y_system,
    u_rl=u_rl,
    avg_rewards=avg_rewards,
    data_min=data_min,
    data_max=data_max,
    reward_fn=reward_fn,
    horizon_trace=horizon_trace,
    mpc_horizons={"Hp": 9, "Hc": 3},
    alpha_log=alpha_log,
    delta_log=delta_log,
    weight_log=weight_log,
    model_low=model_low,
    model_high=model_high,
    weights_low=weights_low,
    weights_high=weights_high,
    mpc_path_or_dir=r"C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Data\mpc_results_dist.pickle",   # add this
    prefix_name="multi_agent_run_dist",
    directory="Results",
    start_episode=6,
    save_pdf=False
)